# 03 — Indexing, Selection, Transformation, and Merging

`.loc`/`.iloc`, boolean masking, string/datetime accessors, and `merge`/`join`/`concat` — the semantics interviewers check because they map directly onto SQL joins and are easy to get subtly wrong.

In [1]:
import pandas as pd
import numpy as np

employees = pd.DataFrame({
    "emp_id": [1, 2, 3, 4],
    "name": ["alice", "bob", "cara", "dan"],
    "dept_id": [10, 10, 20, None],
})
departments = pd.DataFrame({
    "dept_id": [10, 20, 30],
    "dept_name": ["engineering", "sales", "marketing"],
})

## 1. `.loc` vs `.iloc` vs boolean masking

- **`.loc[row_labels, col_labels]`** — label-based. Slicing with `.loc` is **inclusive of the end label** (unlike Python slicing or `.iloc`) — a frequent off-by-one surprise.
- **`.iloc[row_positions, col_positions]`** — purely positional, like NumPy/Python indexing (end-exclusive).
- **Boolean masking** (`df[df["age"] > 30]`) — filters rows by a condition; combine with `.loc` when also selecting columns (`df.loc[df["age"] > 30, "name"]`).
- **`.query("age > 30 and dept == 'eng'")`** — same result as boolean masking, often more readable for multi-condition filters, and doesn't require repeating `df[...]` for every condition.

In [2]:
employees.loc[1:2, ["name", "dept_id"]]   # label-based -- includes row label 2

,name,dept_id
1,bob,10.0
2,cara,20.0


In [3]:
employees.iloc[1:2, [1, 2]]                # position-based -- excludes position 2, like Python slicing

,name,dept_id
1,bob,10.0


In [4]:
employees.query("dept_id == 10")

,emp_id,name,dept_id
0,1,alice,10.0
1,2,bob,10.0


## 2. String and datetime accessors

Vectorized string ops live under `.str` (`.str.lower()`, `.str.contains()`, `.str.extract()`), and date/time component extraction lives under `.dt` (`.dt.year`, `.dt.dayofweek`, `.dt.floor("D")`). Both apply the operation across the whole column in one vectorized call — no `.apply()` needed.

In [5]:
events = pd.DataFrame({
    "user_email": ["Alice@Example.com", "bob@example.com"],
    "event_time": pd.to_datetime(["2024-06-15 14:30:00", "2024-06-16 09:05:00"]),
})

events["domain"] = events["user_email"].str.lower().str.split("@").str[1]
events["hour"] = events["event_time"].dt.hour
events["day_of_week"] = events["event_time"].dt.day_name()
events

,user_email,event_time,domain,hour,day_of_week
0,Alice@Example.com,2024-06-15 14:30:00,example.com,14,Saturday
1,bob@example.com,2024-06-16 09:05:00,example.com,9,Sunday


## 3. `apply`/`map`/`assign` — and when each is appropriate

- **`Series.map(dict_or_func)`** — element-wise, on a single Series; great for value lookups/recoding via a dict.
- **`DataFrame.apply(func, axis=1)`** — row-wise, gets a full `Row` per call; flexible but the slowest option (Python call per row) — reach for a vectorized expression across columns first.
- **`DataFrame.assign(new_col=lambda d: ...)`** — adds columns without mutating the original, chains cleanly in a pipeline of transformations (`df.assign(...).assign(...).query(...)`).

In [6]:
seniority_map = {10: "eng", 20: "sales", 30: "marketing"}

result = (
    employees
    .assign(dept_name=lambda d: d["dept_id"].map(seniority_map))
    .assign(name_upper=lambda d: d["name"].str.upper())
)
result

,emp_id,name,dept_id,dept_name,name_upper
0,1,alice,10.0,eng,ALICE
1,2,bob,10.0,eng,BOB
2,3,cara,20.0,sales,CARA
3,4,dan,NaN,NaN,DAN


## 4. `merge` — pandas' join, mapped directly onto SQL

`pd.merge(left, right, on=key, how=...)` mirrors SQL joins exactly: `how="inner"/"left"/"right"/"outer"`. Two things trip people up:

- **Row multiplication on duplicate keys** — if the right side has multiple rows per key, every matching left row is duplicated once per match (same as SQL) — always sanity-check row counts after a merge if you expected a 1:1 relationship.
- **`indicator=True`** — adds a `_merge` column (`"left_only"`, `"right_only"`, `"both"`) showing exactly where each row's match came from — the fastest way to debug an unexpected row count after a merge, and pandas' equivalent of a `left_anti`/`left_semi` join (filter `_merge == "left_only"`).

In [7]:
merged = employees.merge(departments, on="dept_id", how="left", indicator=True)
merged

,emp_id,name,dept_id,dept_name,_merge
0,1,alice,10.0,engineering,both
1,2,bob,10.0,engineering,both
2,3,cara,20.0,sales,both
3,4,dan,NaN,NaN,left_only


In [8]:
# left_anti equivalent: employees with no matching department (dan, dept_id=NaN)
no_dept = merged[merged["_merge"] == "left_only"]
no_dept

,emp_id,name,dept_id,dept_name,_merge
3,4,dan,NaN,NaN,left_only


## 5. `concat` vs `merge`

`pd.concat([df1, df2])` stacks DataFrames — rows (`axis=0`, the default, like SQL `UNION ALL`) or columns (`axis=1`, aligning by index). It does **not** match rows by a key the way `merge` does — using `concat` when you actually need a key-based join silently produces misaligned rows if the two DataFrames' indexes don't already correspond.

In [9]:
more_employees = pd.DataFrame({"emp_id": [5], "name": ["eve"], "dept_id": [10]})
pd.concat([employees, more_employees], ignore_index=True)

,emp_id,name,dept_id
0,1,alice,10.0
1,2,bob,10.0
2,3,cara,20.0
3,4,dan,NaN
4,5,eve,10.0


## 6. Interview Q&A

1. **"`.loc[0:2]` returned 3 rows, not 2 — why?"** — `.loc` slicing is label-inclusive on both ends; use `.iloc` for the familiar end-exclusive Python slicing behavior.
2. **"After a merge, I have more rows than either input — what happened?"** — a duplicate-key fan-out: one side has multiple rows per key, so matches multiply, exactly like a SQL join; check with `indicator=True` and inspect `value_counts()` on the join key beforehand.
3. **"When would you use `.merge` vs `.concat`?"** — `merge` when rows need to be matched by a key (a join); `concat` when you're stacking datasets that are already row/column-aligned (e.g. appending a new batch of the same schema).
4. **"Why avoid `DataFrame.apply(axis=1)` on a large DataFrame?"** — it calls a Python function once per row, at Python-loop speed; the same logic expressed as vectorized column operations (or `.map()`/`np.where`/`np.select`) runs far faster.

## Summary

- `.loc` = label-based, end-inclusive; `.iloc` = position-based, end-exclusive; `.query()` for readable multi-condition filters.
- `.str`/`.dt` accessors are vectorized — no `.apply()` needed for common string/date operations.
- `merge` = SQL-style key-based join (watch for row fan-out on duplicate keys, use `indicator=True` to debug); `concat` = stacking already-aligned data.
- Next: `04_groupby_aggregation_and_window.ipynb`.